In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [54]:
df = pd.read_csv('data/mammographic_mass.csv')

# lowercase column names
df.columns = df.columns.str.lower().str.replace('-', '_')

# bi_rads is the radiologist's own rating (non-predictive in UCI docs),
# so we drop it to avoid data leakage
df = df.drop(columns=['bi_rads'])
df.head()

,age,shape,margin,density,severity
0,67.0,3.0,5.0,3.0,1
1,43.0,1.0,1.0,NaN,1
2,58.0,4.0,5.0,3.0,1
3,28.0,1.0,1.0,3.0,0
4,74.0,1.0,5.0,NaN,1


In [55]:
df_full_train,df_test=train_test_split(df,test_size=0.2,random_state=1)
df_train,df_val=train_test_split(df_full_train,test_size=0.25,random_state=1)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train=df_train.severity.values
y_val=df_val.severity.values
y_test=df_test.severity.values

del df_train['severity']
del df_val['severity']
del df_test['severity']



In [56]:
print(len(df_train), len(df_val), len(df_test))
print(df_train.columns.tolist())

576 192 193
['age', 'shape', 'margin', 'density']


In [57]:
df_full_train = df_full_train.reset_index(drop=True)

print(df_full_train.isna().sum())
print()
print(df_full_train.describe())

age          3
shape       26
margin      34
density     59
severity     0
dtype: int64

              age       shape      margin     density    severity
count  765.000000  742.000000  734.000000  709.000000  768.000000
mean    55.707190    2.739892    2.844687    2.919605    0.471354
std     14.302107    1.245125    1.580187    0.372837    0.499504
min     19.000000    1.000000    1.000000    1.000000    0.000000
25%     45.000000    2.000000    1.000000    3.000000    0.000000
50%     57.000000    3.000000    3.000000    3.000000    0.000000
75%     66.000000    4.000000    4.000000    3.000000    1.000000
max     96.000000    4.000000    5.000000    4.000000    1.000000


In [58]:
print(df_full_train.severity.value_counts())
print(df_full_train.severity.value_counts(normalize=True))

severity
0    406
1    362
Name: count, dtype: int64
severity
0    0.528646
1    0.471354
Name: proportion, dtype: float64


In [59]:
for col in ['shape', 'margin', 'density']:
    print(col)
    print(df_full_train.groupby(col).severity.agg(['mean', 'count']))
    print()

shape
           mean  count
shape                 
1.0    0.179775    178
2.0    0.179012    162
3.0    0.428571     77
4.0    0.790769    325

margin
            mean  count
margin                 
1.0     0.125000    280
2.0     0.631579     19
3.0     0.608696     92
4.0     0.674208    221
5.0     0.844262    122

density
             mean  count
density                 
1.0      0.454545     11
2.0      0.319149     47
3.0      0.500782    639
4.0      0.416667     12



In [60]:
from sklearn.metrics import mutual_info_score

for col in ['shape', 'margin', 'density']:
    score = mutual_info_score(df_full_train[col].fillna(-1), df_full_train.severity)
    print(col, round(score, 3))
print(df_full_train.groupby('severity').age.describe())    

shape 0.175
margin 0.185
density 0.01
          count       mean        std   min    25%   50%   75%   max
severity                                                            
0         406.0  49.519704  13.274439  19.0  40.25  51.0  59.0  86.0
1         359.0  62.704735  12.027518  31.0  55.00  64.0  71.0  96.0


In [61]:
print(df_full_train.severity.value_counts())

severity
0    406
1    362
Name: count, dtype: int64


In [62]:
categorical = ['shape', 'margin', 'density']
numerical = ['age']

age_median = df_train.age.median()

def prepare(df):
    df = df.copy()
    df['age'] = df['age'].fillna(age_median)
    for col in categorical:
        # missing -> 'unknown'
        df[col] = df[col].apply(lambda v: 'unknown' if pd.isna(v) else str(int(v)))
    return df


df_train = prepare(df_train)
df_val = prepare(df_val)
df_test = prepare(df_test)

print(age_median)
print(df_train.isna().sum())
print(df_train.head())

57.0
age        0
shape      0
margin     0
density    0
dtype: int64
    age shape margin  density
0  77.0     4      5        3
1  50.0     3      2        3
2  63.0     2      1  unknown
3  35.0     1      1        2
4  56.0     4      4        3


In [63]:
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
dv=DictVectorizer(sparse=False)
X_train=dv.fit_transform(train_dicts)
X_val=dv.transform(val_dicts)
print(X_train.shape, X_val.shape)
print(dv.get_feature_names_out())

(576, 17) (192, 17)
['age' 'density=1' 'density=2' 'density=3' 'density=4' 'density=unknown'
 'margin=1' 'margin=2' 'margin=3' 'margin=4' 'margin=5' 'margin=unknown'
 'shape=1' 'shape=2' 'shape=3' 'shape=4' 'shape=unknown']


In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
y_pred=model.predict_proba(X_val)[:,1]
roc_auc_score(y_val,y_pred)

0.8797653958944281

In [65]:
from sklearn.tree import DecisionTreeClassifier
for d in [1, 2, 3, 4, 5, 6, 10, 15, 20, None]:
    for s in [1, 2, 5, 10, 15, 20, 50, 100]:
        dt=DecisionTreeClassifier(max_depth=d, min_samples_leaf=s, random_state=1)
        dt.fit(X_train,y_train)
        y_pred=dt.predict_proba(X_val)[:,1]
        auc=roc_auc_score(y_val,y_pred)
        print(f"max_depth: {str(d):4}, min_samples_leaf: {s:3} ---> AUC: {auc:.4f}")
    




max_depth: 1   , min_samples_leaf:   1 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:   2 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:   5 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:  10 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:  15 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:  20 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf:  50 ---> AUC: 0.7193
max_depth: 1   , min_samples_leaf: 100 ---> AUC: 0.7193
max_depth: 2   , min_samples_leaf:   1 ---> AUC: 0.8234
max_depth: 2   , min_samples_leaf:   2 ---> AUC: 0.8234
max_depth: 2   , min_samples_leaf:   5 ---> AUC: 0.8234
max_depth: 2   , min_samples_leaf:  10 ---> AUC: 0.8234
max_depth: 2   , min_samples_leaf:  15 ---> AUC: 0.8234
max_depth: 2   , min_samples_leaf:  20 ---> AUC: 0.8258
max_depth: 2   , min_samples_leaf:  50 ---> AUC: 0.8231
max_depth: 2   , min_samples_leaf: 100 ---> AUC: 0.8276
max_depth: 3   , min_samples_leaf:   1 ---> AUC: 0.8484
max_depth: 3   , min_samples_leaf:   2 ---> AUC:

In [66]:
from sklearn.ensemble import RandomForestClassifier
best_auc=0
best_params={}
for n in [10, 30, 50, 100]:
    for s  in [3,5,10,20,None]:
        for d in [1,5,10,20]:
            rf=RandomForestClassifier(n_estimators=n, max_depth=s,min_samples_leaf=d ,random_state=1)
            rf.fit(X_train,y_train)
            y_pred=rf.predict_proba(X_val)[:,1]
            auc=roc_auc_score(y_val,y_pred)
            if auc>best_auc:
                best_auc=auc
                best_params={'n_estimators':n, 'max_depth':d, 'min_samples_leaf':s}
            print(f"n_estimators: {n:3} ---> AUC: {auc:.4f}",f"best_params: {best_params}")


n_estimators:  10 ---> AUC: 0.8867 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8834 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8765 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8748 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8743 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8738 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8768 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8774 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 0.8439 best_params: {'n_estimators': 10, 'max_depth': 1, 'min_samples_leaf': 3}
n_estimators:  10 ---> AUC: 

In [67]:
features =list(dv.get_feature_names_out())
dtrain=xgb.DMatrix(X_train,label=y_train)
dval=xgb.DMatrix(X_val,label=y_val)

learning_rate_values = [0.05,0.01, 0.1, 0.2, 0.3]
depth_values = [ 3, 4, 6]
min_child_weight_values = [1, 5]
best_auc=0
best_params={}
for learning_rate in learning_rate_values:
    for depth in depth_values:
        for min_child_weight in min_child_weight_values:
            params={
                'learning_rate':learning_rate,
                'max_depth':depth,
                'min_child_weight':min_child_weight,
                'objective':'binary:logistic',
                'eval_metric':'auc',
                'seed':1
            }
            model=xgb.train(params, dtrain, num_boost_round=200)
            y_pred=model.predict(dval)
            auc=roc_auc_score(y_val,y_pred)
            if auc>best_auc:
                best_auc=auc
                best_params=params
            print(f"learning_rate: {learning_rate}, depth: {depth}, min_child_weight: {min_child_weight} ---> AUC: {auc:.4f}")

learning_rate: 0.05, depth: 3, min_child_weight: 1 ---> AUC: 0.8756
learning_rate: 0.05, depth: 3, min_child_weight: 5 ---> AUC: 0.8637
learning_rate: 0.05, depth: 4, min_child_weight: 1 ---> AUC: 0.8674
learning_rate: 0.05, depth: 4, min_child_weight: 5 ---> AUC: 0.8574
learning_rate: 0.05, depth: 6, min_child_weight: 1 ---> AUC: 0.8579
learning_rate: 0.05, depth: 6, min_child_weight: 5 ---> AUC: 0.8541
learning_rate: 0.01, depth: 3, min_child_weight: 1 ---> AUC: 0.8750
learning_rate: 0.01, depth: 3, min_child_weight: 5 ---> AUC: 0.8682
learning_rate: 0.01, depth: 4, min_child_weight: 1 ---> AUC: 0.8728
learning_rate: 0.01, depth: 4, min_child_weight: 5 ---> AUC: 0.8632
learning_rate: 0.01, depth: 6, min_child_weight: 1 ---> AUC: 0.8542
learning_rate: 0.01, depth: 6, min_child_weight: 5 ---> AUC: 0.8669
learning_rate: 0.1, depth: 3, min_child_weight: 1 ---> AUC: 0.8655
learning_rate: 0.1, depth: 3, min_child_weight: 5 ---> AUC: 0.8573
learning_rate: 0.1, depth: 4, min_child_weight: 1 

In [69]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
df_cv=pd.concat([df_train, df_val])
cv_dicts=df_cv[categorical + numerical].to_dict(orient='records')
y_cv=np.concatenate([y_train, y_val])
models={
    'logistic_regression': LogisticRegression(max_iter=1000),
    'decision_tree': DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=1),
    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=3, min_samples_leaf=1, random_state=1,n_jobs=-1),
    'xgboost': XGBClassifier(learning_rate=0.05, max_depth=3, min_child_weight=1,n_estimators=200,random_state=1)
}
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
for C in [0.01, 0.1, 1, 10, 100]:
    pipeline = make_pipeline(DictVectorizer(sparse=False), LogisticRegression(C=C, max_iter=1000))
    scores = cross_val_score(pipeline, cv_dicts, y_cv, scoring='roc_auc', cv=cv, n_jobs=-1)
    print(f"C={C}: mean AUC={scores.mean():.4f}, std={scores.std():.4f}")
for model_name, model in models.items():
    pipeline=make_pipeline(DictVectorizer(sparse=False), model)     
    scores=cross_val_score(pipeline, cv_dicts, y_cv, scoring='roc_auc', cv=cv, n_jobs=-1)
    print(f"{model_name}: mean AUC={scores.mean():.4f}, std={scores.std():.4f}")

# ### Model Selection

C=0.01: mean AUC=0.8465, std=0.0258
C=0.1: mean AUC=0.8746, std=0.0273
C=1: mean AUC=0.8709, std=0.0288
C=10: mean AUC=0.8698, std=0.0286
C=100: mean AUC=0.8695, std=0.0288
logistic_regression: mean AUC=0.8709, std=0.0288
decision_tree: mean AUC=0.8412, std=0.0323
random_forest: mean AUC=0.8664, std=0.0305
xgboost: mean AUC=0.8569, std=0.0295


Full training

In [71]:
# final model: logistic regression (C=0.1), trained on train + validation
df_full_train = pd.concat([df_train, df_val])

full_train_dicts = df_full_train[categorical + numerical].to_dict(orient='records')
test_dicts = df_test[categorical + numerical].to_dict(orient='records')
y_full_train = np.concatenate([y_train, y_val])

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(full_train_dicts)
X_test = dv.transform(test_dicts)

final_model = LogisticRegression(C=0.1, max_iter=1000)
final_model.fit(X_full_train, y_full_train)

y_pred_test = final_model.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_pred_test)
print(f"Final AUC on test set: {final_auc:.4f}")

Final AUC on test set: 0.8438


In [73]:
#Save model
import pickle
output_file = 'model.bin'
with open(output_file, 'wb') as f_out:
    pickle.dump((dv, final_model, age_median), f_out)
print(f"Model saved to {output_file}")


Model saved to model.bin
